# Age Estimation
Compute stellar ages using MIST, Dartmouth, GARSTEC, and YREC grids.
Follows the multi-model strategy from Pinsonneault et al. (2025).
Cluster ages use a 60/40 blend of cluster mean mass and individual mass.
Error propagation per Section 2.13.3: perturb mass and metallicity by +/-1 sigma.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

AGE_CAP = 13.7  # Gyr, age of universe
BLEND = 0.60    # 60% cluster mean mass, 40% individual

# Grid paths (update these to your local paths)
GRID_PATHS = {
    'MIST': '../../grids_real/MIST/mist/mist_eep.pqt',
    'Dartmouth': '../../grids_real/Dartmouth/dartmouth/dartmouth_eep.pqt',
    'GARSTEC': '../../grids_real/GARSTEC/garstec/garstec_eep.pqt',
    'YREC': '../../grids_real/YREC/yrec/yrec_eep.pqt',
}

In [ ]:
def load_grid(name, path):
    g = pd.read_parquet(path).reset_index()
    out = pd.DataFrame()
    out['initial_mass'] = g['initial_mass']
    out['initial_met'] = g['initial_met']
    if name == 'MIST':
        out['age_gyr'] = g['star_age'] / 1e9
        out['log_Teff'] = g['log_Teff']
        out['log_g'] = g['log_g']
    elif name == 'Dartmouth':
        out['age_gyr'] = g['Age (yrs)'] / 1e9
        out['log_Teff'] = g['Log T']
        out['log_g'] = g['Log g']
    elif name == 'GARSTEC':
        out['age_gyr'] = g['Age(Myr)'] / 1e3
        out['log_Teff'] = np.log10(g['Teff'])
        out['log_g'] = g['logg']
    elif name == 'YREC':
        out['age_gyr'] = g['Age(Gyr)']
        out['log_Teff'] = g['Log Teff(K)']
        out['log_g'] = g['logg']
    return out[(out['log_g'] < 3.5) & (out['log_Teff'] < np.log10(6000)) & (out['age_gyr'] > 0.05)]

def get_age(mass, teff, feh, grid, mass_tol=0.15, feh_tol=0.3, teff_tol=200):
    log_teff = np.log10(teff)
    mask = ((np.abs(grid['initial_mass'] - mass) < mass_tol) &
            (np.abs(grid['initial_met'] - feh) < feh_tol) &
            (np.abs(grid['log_Teff'] - log_teff) < np.log10(1 + teff_tol / teff)))
    sub = grid[mask]
    if len(sub) < 3:
        sub = grid[(np.abs(grid['initial_mass'] - mass) < mass_tol * 2) &
                   (np.abs(grid['initial_met'] - feh) < feh_tol * 2)]
    if len(sub) == 0:
        return np.nan
    chi2 = ((sub['initial_mass'] - mass) / 0.05)**2 + \
           ((sub['initial_met'] - feh) / 0.2)**2 + \
           ((sub['log_Teff'] - log_teff) / 0.01)**2
    w = np.exp(-0.5 * chi2)
    w /= w.sum()
    return (sub['age_gyr'] * w).sum()

grids = {n: load_grid(n, p) for n, p in GRID_PATHS.items()}
print('Grids loaded.')

In [ ]:
fsr = pd.read_csv('../data/seismic_results.csv')
multi_clusters = ['Casado_Alessi_1', 'NGC_752', 'Theia_6046', 'Theia_844']
cmm = {cl: fsr[fsr['cluster'] == cl]['M_seis'].mean() for cl in multi_clusters}

results = []
for _, row in fsr.iterrows():
    tic = int(row['TICID'])
    cl = row['cluster']
    m_ind = row['M_seis']
    teff = row['teff']
    feh = row['feh']
    m_err = row['M_err_up'] if 'M_err_up' in row else m_ind * 0.16
    
    m_use = BLEND * cmm[cl] + (1 - BLEND) * m_ind if cl in multi_clusters else m_ind
    
    # Central ages from all 4 grids
    ages = {gn: get_age(m_use, teff, feh, g) for gn, g in grids.items()}
    ages = {k: min(v, AGE_CAP) if not np.isnan(v) else np.nan for k, v in ages.items()}
    age_mist = ages.get('MIST', np.nan)
    
    # sigma_obs: full +/-1 sigma mass AND metallicity perturbation (Section 2.13.3)
    amp = get_age(m_use + m_err, teff, feh, grids['MIST'])
    amm = get_age(m_use - m_err, teff, feh, grids['MIST'])
    afp = get_age(m_use, teff, feh + 0.1, grids['MIST'])
    afm = get_age(m_use, teff, feh - 0.1, grids['MIST'])
    sm = abs(amp - amm) / 2 if not np.isnan(amp) and not np.isnan(amm) else 0.3
    sf = abs(afp - afm) / 2 if not np.isnan(afp) and not np.isnan(afm) else 0.1
    sigma_obs = np.sqrt(sm**2 + sf**2)
    
    # sigma_model: grid to grid spread
    valid = [a for a in ages.values() if not np.isnan(a)]
    sigma_model = np.std(valid) if len(valid) > 1 else 0.5
    age_std = np.sqrt(sigma_obs**2 + sigma_model**2)
    
    results.append({'TICID': tic, 'cluster': cl, 'age_gyr': age_mist, 'age_std': age_std,
        'sigma_obs': sigma_obs, 'sigma_model': sigma_model,
        'age_MIST': ages['MIST'], 'age_Dartmouth': ages['Dartmouth'],
        'age_GARSTEC': ages['GARSTEC'], 'age_YREC': ages['YREC']})

res = pd.DataFrame(results)
res.to_csv('../data/age_results.csv', index=False)

# Cluster summary (Eq 2 and 4)
for cl in sorted(res['cluster'].unique()):
    sub = res[res['cluster'] == cl].dropna(subset=['age_gyr'])
    n = len(sub)
    t = sub['age_gyr'].mean()
    s_star = sub['age_gyr'].std() if n > 1 else 0
    sigma = np.sqrt((1/n**2) * (sub['age_std']**2).sum() + s_star**2) if n > 1 else sub['age_std'].values[0]
    print(f'{cl}: {t:.3f} +/- {sigma:.3f} Gyr (n={n})')